# GRN inference

Infers transcription factor (TF) → target gene edges for two stimulus contexts, K562 + TNF-α and THP1 + LPS. Each context uses three steps:

1. **Rank TFs** (`cp.tl.grn.rank_regulators`): signed diffusion over the OmniPath signaling network from the context's upstream signal. TFs (human TFs with OmniPath transcriptional targets) absorb the mass that reaches them and are ranked by the net activating mass.
2. **Score target genes** (`cp.tl.grn.score_targets`): convert the context's tradeSeq association p-values into local FDRs (fdrtool / beta-uniform mixture) over protein-coding genes.
3. **Infer edges** (`cp.tl.grn.infer`): weight each TF → target edge by the TF's priority and the target's 1 - local FDR, then rank the edges.

Results for each context are written to `outputs/<context>/`.


In [1]:
from pathlib import Path

import chronopack as cp

HUMAN_TFS = "/new-stg/home/alex/ref/gene_sets/hs_tfs.txt"
LIGAND_RECEPTORS = "/new-stg/home/alex/ref/cellchatdb/hs/interaction_input_CellChatDB.csv"
PROTEIN_CODING = "/new-stg/home/alex/tmp/grn_inference_v2/reproduction/inputs/external/ref/genome/hg38/annotations/protein_coding_genes.csv"
ANALYSES = Path("/new-stg/home/alex/tmp/grn_inference_v2/reproduction/inputs/external/projects/chrono_seq/analyses")

## K562 + TNF-α

TFs are ranked downstream of the TNF ligand (seeded at its receptors); target genes come from the K562 tradeSeq association test.

In [2]:
tfs_k562 = cp.tl.grn.rank_regulators(
    source="TNF",
    is_ligand=True,
    human_tf_path=HUMAN_TFS,
    ligand_receptor_path=LIGAND_RECEPTORS,
)
tfs_k562.head(10)

/new-stg/home/alex/miniconda3/envs/chronopack/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,TF,abs_pos,abs_neg,support,net,score,priority_act,priority_abs
0,RELA,0.016358,0.002585,0.018943,0.013774,0.727104,0.013774,0.013774
1,NFKB1,0.015486,0.005250,0.020736,0.010237,0.493673,0.010237,0.010237
2,STAT3,0.010025,0.002885,0.012910,0.007140,0.553066,0.007140,0.007140
3,NFKB2,0.004822,0.000570,0.005392,0.004252,0.788603,0.004252,0.004252
4,E2F1,0.004225,0.000504,0.004729,0.003721,0.786801,0.003721,0.003721
5,IRF5,0.003468,0.000273,0.003741,0.003195,0.854214,0.003195,0.003195
6,JUN,0.003080,0.000419,0.003499,0.002661,0.760492,0.002661,0.002661
7,SMAD1,0.003490,0.001058,0.004548,0.002432,0.534827,0.002432,0.002432
8,IRF7,0.002555,0.000205,0.002760,0.002350,0.851661,0.002350,0.002350
9,IRF3,0.002123,0.000280,0.002403,0.001843,0.766786,0.001843,0.001843


In [3]:
targets_k562 = cp.tl.grn.score_targets(
    ANALYSES / "k562-tnfa.hg38.rep/intmd/sc-K562-1e4_to_tradeSeq/asso_df.1trajectory.lfdr.csv",
    valid_targets=PROTEIN_CODING,
)
targets_k562.summary

,method,n,below_0p01,below_0p05,below_0p10,below_0p20,exact_one,median,q10,q90
0,stored_lfdr,11394,394,521,642,794,0,1.000000,0.534806,1.000000
1,lfdr_fdrtool_fndr,11394,394,521,653,824,9157,1.000000,0.424836,1.000000
2,lfdr_fdrtool_pct0,11394,394,521,653,824,9290,1.000000,0.443377,1.000000
3,lfdr_fdrtool_locfdr,11394,394,521,653,824,9157,1.000000,0.424836,1.000000
4,bh_q,11394,539,818,1015,1353,0,1.000000,0.144928,1.000000
5,lfdr_like_bh_q,11394,539,818,1015,1353,0,1.000000,0.144928,1.000000
6,lfdr_bum,11394,381,523,605,734,0,0.969338,0.552937,0.980219
7,lfdr_like_logp_cap10,11394,100,107,115,136,0,0.977374,0.816150,0.999732


In [4]:
grn_k562 = cp.tl.grn.infer(targets_k562, tfs_k562, valid_targets=PROTEIN_CODING, name="ChronoPack_K562")
grn_k562.edges.head(10)

,method,regulator,target,score,rank,percentile
0,ChronoPack_K562,RELA,NFKBIA,1.962572e+06,1,1.000000
1,ChronoPack_K562,RELA,CXCL3,1.962572e+06,2,1.000000
2,ChronoPack_K562,RELA,CD69,1.962572e+06,3,0.999999
3,ChronoPack_K562,RELA,CD83,1.962572e+06,4,0.999999
4,ChronoPack_K562,RELA,ARL4A,1.962572e+06,5,0.999998
5,ChronoPack_K562,RELA,ICAM1,1.962572e+06,6,0.999998
6,ChronoPack_K562,RELA,CXCL8,1.962572e+06,7,0.999997
7,ChronoPack_K562,RELA,BTG2,1.962572e+06,8,0.999997
8,ChronoPack_K562,RELA,BIRC3,1.962572e+06,9,0.999997
9,ChronoPack_K562,RELA,DDIT4,1.962572e+06,10,0.999996


In [5]:
targets_k562.write("outputs/K562", prefix="target_scores")
tfs_k562.to_csv("outputs/K562/tf_ranking.csv", index=False)
grn_k562.write("outputs/K562", prefix="grn")

{'edges_csv': 'outputs/K562/grn.edges.csv',
 'metadata': 'outputs/K562/grn.metadata.json',
 'regulators': 'outputs/K562/grn.regulators.csv',
 'target_scores': 'outputs/K562/grn.target_scores.csv'}

## THP1 + LPS

TFs are ranked downstream of the TLR4 receptor; target genes come from the THP1 tradeSeq association test.

In [6]:
tfs_thp1 = cp.tl.grn.rank_regulators(
    source="TLR4",
    human_tf_path=HUMAN_TFS,
    ligand_receptor_path=LIGAND_RECEPTORS,
)
tfs_thp1.head(10)

,TF,abs_pos,abs_neg,support,net,score,priority_act,priority_abs
0,NFKB1,0.025628,0.003860,0.029488,0.021768,0.738185,0.021768,0.021768
1,RELA,0.025501,0.003972,0.029473,0.021530,0.730480,0.021530,0.021530
2,IRF7,0.016211,0.000504,0.016715,0.015707,0.939704,0.015707,0.015707
3,IRF5,0.006967,0.000330,0.007297,0.006637,0.909528,0.006637,0.006637
4,IRF3,0.006707,0.000317,0.007024,0.006389,0.909730,0.006389,0.006389
5,REL,0.006277,0.000181,0.006458,0.006096,0.943872,0.006096,0.006096
6,ESR1,0.005465,0.000314,0.005779,0.005151,0.891387,0.005151,0.005151
7,JUN,0.005692,0.000586,0.006278,0.005106,0.813267,0.005106,0.005106
8,STAT1,0.005242,0.000377,0.005619,0.004866,0.865933,0.004866,0.004866
9,TP53,0.007493,0.003038,0.010531,0.004455,0.423074,0.004455,0.004455


In [7]:
targets_thp1 = cp.tl.grn.score_targets(
    ANALYSES / "thp1-k562-lps.hg38.rep/intmd/sc-THP1_to_tradeSeq/asso_df.1trajectory.lfdr.csv",
    valid_targets=PROTEIN_CODING,
)
targets_thp1.summary

,method,n,below_0p01,below_0p05,below_0p10,below_0p20,exact_one,median,q10,q90
0,stored_lfdr,12324,139,139,139,139,11735,1.000000,1.000000,1.000000
1,lfdr_fdrtool_fndr,12324,123,123,123,123,11179,1.000000,1.000000,1.000000
2,lfdr_fdrtool_pct0,12324,0,0,0,0,11662,1.000000,1.000000,1.000000
3,lfdr_fdrtool_locfdr,12324,123,123,123,123,11179,1.000000,1.000000,1.000000
4,bh_q,12324,260,368,454,555,0,1.000000,0.746055,1.000000
5,lfdr_like_bh_q,12324,260,368,454,555,0,1.000000,0.746055,1.000000
6,lfdr_bum,12324,201,239,262,312,0,0.994538,0.957883,0.995923
7,lfdr_like_logp_cap10,12324,91,96,101,117,16,0.986022,0.887309,0.999968


In [8]:
grn_thp1 = cp.tl.grn.infer(targets_thp1, tfs_thp1, valid_targets=PROTEIN_CODING, name="ChronoPack_THP1")
grn_thp1.edges.head(10)

,method,regulator,target,score,rank,percentile
0,ChronoPack_THP1,NFKB1,CXCL8,2.183429e+06,1,1.000000
1,ChronoPack_THP1,NFKB1,CCL4,2.183429e+06,2,1.000000
2,ChronoPack_THP1,NFKB1,TNFAIP2,2.183429e+06,3,0.999999
3,ChronoPack_THP1,NFKB1,CCL4L2,2.183429e+06,4,0.999999
4,ChronoPack_THP1,NFKB1,NFKBIA,2.183429e+06,5,0.999999
5,ChronoPack_THP1,NFKB1,SOD2,2.183429e+06,6,0.999998
6,ChronoPack_THP1,NFKB1,CCL3,2.183429e+06,7,0.999998
7,ChronoPack_THP1,NFKB1,CXCL3,2.183429e+06,8,0.999997
8,ChronoPack_THP1,NFKB1,TNFAIP3,2.183429e+06,9,0.999997
9,ChronoPack_THP1,NFKB1,CXCL1,2.183429e+06,10,0.999997


In [9]:
targets_thp1.write("outputs/THP1", prefix="target_scores")
tfs_thp1.to_csv("outputs/THP1/tf_ranking.csv", index=False)
grn_thp1.write("outputs/THP1", prefix="grn")

{'edges_csv': 'outputs/THP1/grn.edges.csv',
 'metadata': 'outputs/THP1/grn.metadata.json',
 'regulators': 'outputs/THP1/grn.regulators.csv',
 'target_scores': 'outputs/THP1/grn.target_scores.csv'}